# Amylogram Amyloid Probability

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
WT_COLOR = "#4ff7c0"
C_DESTAB = "#3b82f6"
C_NEUTRAL = "#64748b"
C_STAB = "#ef4444"
THRESH = 0.001  # threshold — lower due to data discreteness

# ── Loading ──────────────────────────────────────────────────
df = pd.read_csv("../predictions/amylogram/AmyloGram_results_ed.csv")
df["short"] = df["Input name"].str.replace(r"_Abeta_?42$", "", regex=True)
wt_p = df[df["Input name"] == "Wildtype_Abeta_42"]["Amyloid probability"].iloc[0]
df["ΔP"] = df["Amyloid probability"] - wt_p

In [ ]:
def pc(d):
    if d < -THRESH:
        return C_DESTAB
    if d > THRESH:
        return C_STAB
    return C_NEUTRAL


n = len(df)
fs = max(4.2, min(6.5, 8 * 20 / n))

# Unique values — for annotation
uniq = sorted(df["Amyloid probability"].unique())
print(f"Number of unique probability values: {len(uniq)}")
for v in uniq:
    grp = df[df["Amyloid probability"] == v]["short"].tolist()
    dp = v - wt_p
    print(
        f"  {v:.6f}  (Δ={dp:+.6f})  n={len(grp)}: {', '.join(grp[:5])}{'...' if len(grp) > 5 else ''}"
    )

In [ ]:
# PLOT 1 — Lollipop: ΔProbability

df_lol = df.sort_values("ΔP", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 8), facecolor=BG)
ax.set_facecolor(SURFACE)

for i in range(n):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color="white", alpha=0.018, zorder=0)

# WT line
ax.axvline(0, color=WT_COLOR, lw=1.0, linestyle="--", alpha=0.55)
ax.text(
    0,
    n + 0.3,
    f"WT\n{wt_p:.4f}",
    ha="center",
    va="bottom",
    color=WT_COLOR,
    fontsize=6.5,
    fontfamily="monospace",
    fontweight="bold",
)

# Vertical lines at unique Δ values
for v in uniq:
    d = v - wt_p
    if abs(d) > THRESH:
        ax.axvline(d, color="white", lw=0.35, alpha=0.12, linestyle=":")

# Zones
ax.axvspan(df["ΔP"].min() - 0.005, -THRESH, color=C_DESTAB, alpha=0.05)
ax.axvspan(THRESH, df["ΔP"].max() + 0.005, color=C_STAB, alpha=0.04)

# Lollipops
for i, (_, row) in enumerate(df_lol.iterrows()):
    c = pc(row["ΔP"])
    ax.plot([0, row["ΔP"]], [i, i], color=c, lw=0.8, alpha=0.45)
    ax.scatter(row["ΔP"], i, color=c, s=22, zorder=3, linewidths=0)
    # Numerical value only for non-neutral
    if abs(row["ΔP"]) > THRESH:
        xoff = row["ΔP"] + (0.0008 if row["ΔP"] >= 0 else -0.0008)
        ax.text(
            xoff,
            i,
            f"{row['ΔP']:+.4f}",
            va="center",
            ha="left" if row["ΔP"] >= 0 else "right",
            fontsize=5,
            fontfamily="monospace",
            color=c,
            alpha=0.8,
        )

# Annotate unique levels at the top
for v in uniq:
    d = v - wt_p
    if abs(d) > THRESH:
        c = pc(d)
        n_grp = (df["Amyloid probability"] == v).sum()
        ax.text(
            d,
            n + 0.8,
            f"n={n_grp}",
            ha="center",
            va="bottom",
            color=c,
            fontsize=6,
            fontfamily="monospace",
            alpha=0.75,
        )

ax.set_yticks(range(n))
ax.set_yticklabels(df_lol["short"].values, fontsize=fs, fontfamily="monospace")
for tick, (_, row) in zip(ax.get_yticklabels(), df_lol.iterrows()):
    tick.set_color(pc(row["ΔP"]))
    tick.set_alpha(0.9)

ax.tick_params(axis="y", length=0, pad=3)
ax.tick_params(axis="x", colors=MUTED, labelsize=7)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xlim(df["ΔP"].min() - 0.012, df["ΔP"].max() + 0.018)
ax.set_ylim(-1, n + 1.5)

n_d = (df_lol["ΔP"] < -THRESH).sum()
n_n = (df_lol["ΔP"].abs() <= THRESH).sum()
n_s = (df_lol["ΔP"] > THRESH).sum()
lp = [
    mpatches.Patch(
        facecolor=C_DESTAB, label=f"Decrease amyloidogenicity (Δ<0)  · n={n_d}"
    ),
    mpatches.Patch(facecolor=C_NEUTRAL, label=f"= WT (neutral)  · n={n_n}"),
    mpatches.Patch(
        facecolor=C_STAB, label=f"Increase amyloidogenicity (Δ>0)  · n={n_s}"
    ),
]
ax.legend(
    handles=lp,
    loc="lower right",
    frameon=True,
    framealpha=0.15,
    edgecolor=MUTED,
    facecolor=SURFACE,
    fontsize=7,
    labelcolor=TEXT,
    handlelength=0.9,
)

ax.set_xlabel(
    "Δ Amyloid Probability  (mutant − WT)\n",
    color=MUTED,
    fontsize=8,
    fontfamily="monospace",
    labelpad=8,
)
ax.set_title(
    f"AmyloGram  ·  Δ Probability vs Wildtype\n"
    f"WT = {wt_p:.6f}  ·  {len(uniq)} unique values out of {n} variants",
    color="white",
    fontsize=11,
    fontfamily="monospace",
    fontweight="bold",
    pad=12,
)

plt.tight_layout()
plt.savefig(
    "amylogram_lollipop.png",
    dpi=200,
    bbox_inches="tight",
    facecolor=BG,
    edgecolor="none",
)
print("Saved: amylogram_lollipop.png")
plt.close()

In [ ]:
# PLOT 2 — Grouped dot plot by unique levels
# Shows which variants fall into each discrete level

# Group by level, sort by ΔP within each group
df_grp = df.sort_values(["Amyloid probability", "short"], ascending=[True, True])
level_order = sorted(uniq)  # from min to max

# Level colors
level_colors = {v: pc(v - wt_p) for v in level_order}
level_labels = {v: f"{v:.4f}\n(Δ={v - wt_p:+.4f})" for v in level_order}

fig, axes = plt.subplots(
    1,
    len(uniq),
    figsize=(len(uniq) * 3.2 + 1, 7),
    facecolor=BG,
    gridspec_kw={"wspace": 0.12},
)

for ax_i, lv in enumerate(level_order):
    ax = axes[ax_i]
    ax.set_facecolor(SURFACE)

    grp = df_grp[df_grp["Amyloid probability"] == lv].reset_index(drop=True)
    c = level_colors[lv]
    n_g = len(grp)

    for i, (_, row) in enumerate(grp.iterrows()):
        is_wt = row["Input name"] == "Wildtype_Abeta_42"
        dot_c = WT_COLOR if is_wt else c
        sz = 90 if is_wt else 55
        ax.scatter(
            0,
            i,
            color=dot_c,
            s=sz,
            zorder=3,
            linewidths=0,
            marker="*" if is_wt else "o",
        )
        ax.text(
            0.06,
            i,
            row["short"],
            va="center",
            ha="left",
            fontsize=6.2,
            fontfamily="monospace",
            color=dot_c,
            alpha=0.9,
        )

    # Box header
    box_color = c
    ax.axhspan(n_g - 0.1, n_g + 0.9, color=box_color, alpha=0.12, zorder=0)
    ax.text(
        0,
        n_g + 0.4,
        f"{lv:.4f}",
        ha="center",
        va="center",
        fontsize=8.5,
        fontfamily="monospace",
        color=box_color,
        fontweight="bold",
    )
    ax.text(
        0,
        n_g + 0.05,
        f"Δ = {lv - wt_p:+.4f}",
        ha="center",
        va="top",
        fontsize=6.5,
        fontfamily="monospace",
        color=box_color,
        alpha=0.8,
    )

    ax.set_xlim(-0.3, 2.5)
    ax.set_ylim(-0.7, n_g + 1.0)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    # Count at bottom
    ax.text(
        0,
        -0.5,
        f"n = {n_g}",
        ha="center",
        va="center",
        fontsize=7,
        fontfamily="monospace",
        color=MUTED,
    )

fig.suptitle(
    "AmyloGram  ·  Distribution of variants by discrete probability levels\n"
    "Left — least amyloidogenic, right — most amyloidogenic",
    color="white",
    fontsize=12,
    fontfamily="monospace",
    fontweight="bold",
    y=1.02,
)

lp2 = [
    mpatches.Patch(facecolor=C_DESTAB, label="Below WT"),
    mpatches.Patch(facecolor=C_NEUTRAL, label="= WT"),
    mpatches.Patch(facecolor=C_STAB, label="Above WT"),
    plt.scatter([], [], marker="*", s=80, color=WT_COLOR, label="Wildtype"),
]
fig.legend(
    handles=lp2,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=8,
    labelcolor=TEXT,
    bbox_to_anchor=(0.5, -0.03),
)

plt.savefig(
    "amylogram_groups.png", dpi=200, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved: amylogram_groups.png")
plt.close()